# London Daily Temperature Prediction

---

This notebook builds an end-to-end machine learning pipeline to predict the **daily mean temperature in London** using historical weather observations.

| | |
|---|---|
| **Dataset** | London Weather Data (ECA&D), 1979–2020 |
| **Target** | Daily mean temperature (°C) |
| **Models** | Ridge · Lasso · Random Forest · Gradient Boosting · SVR |
| **Tracking** | MLflow |

---

**Sections**
1. [Imports & Configuration](#1)
2. [Data Loading & Overview](#2)
3. [Exploratory Data Analysis](#3)
4. [Feature Engineering](#4)
5. [Train / Test Split](#5)
6. [Model Training & Experiment Tracking](#6)
7. [Model Comparison](#7)
8. [Best Model Evaluation](#8)
9. [Feature Importance](#9)
10. [Summary](#10)

<a id='1'></a>
## 1 · Imports & Configuration

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import mlflow
import mlflow.sklearn

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from data_preprocessing import load_data, engineer_features, clean_data, get_X_y
from models import build_pipelines

# ── Global plot style ────────────────────────────────────────────────────────
PALETTE   = 'coolwarm'
ACCENT    = '#2E86AB'
HIGHLIGHT = '#E84855'

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'axes.titlesize': 12,
})

<a id='2'></a>
## 2 · Data Loading & Overview

In [ ]:
df_raw = load_data('../data/london_weather.csv')

print(f"Records : {len(df_raw):,}")
print(f"Date range: {df_raw['date'].min().date()}  →  {df_raw['date'].max().date()}")
print(f"Columns : {df_raw.shape[1]}")
df_raw.head(8)

In [ ]:
df_raw.describe().round(2)

In [ ]:
# Missing value summary
missing = (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
missing = missing[missing > 0].sort_values(ascending=False).rename('Missing (%)')

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(missing.index, missing.values, color=ACCENT, edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=9)
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Values by Feature')
ax.set_xlim(0, missing.max() * 1.3)
plt.tight_layout()
plt.savefig('../outputs/missing_values.png', bbox_inches='tight')
plt.show()

<a id='3'></a>
## 3 · Exploratory Data Analysis

In [ ]:
# ── 3.1  Temperature over time ───────────────────────────────────────────────
df_raw['year']  = df_raw['date'].dt.year
df_raw['month'] = df_raw['date'].dt.month

yearly_avg = df_raw.groupby('year')['mean_temp'].mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'hspace': 0.45})

# Daily series
axes[0].plot(df_raw['date'], df_raw['mean_temp'],
             linewidth=0.5, alpha=0.6, color=ACCENT, label='Daily')
axes[0].plot(df_raw.set_index('date')['mean_temp'].rolling(365).mean(),
             linewidth=2, color=HIGHLIGHT, label='365-day rolling mean')
axes[0].set_title('Daily Mean Temperature — London (1979–2020)')
axes[0].set_ylabel('Temperature (°C)')
axes[0].legend(frameon=False)

# Yearly average
axes[1].fill_between(yearly_avg.index, yearly_avg.values,
                     alpha=0.25, color=ACCENT)
axes[1].plot(yearly_avg.index, yearly_avg.values,
             marker='o', markersize=4, color=ACCENT, linewidth=1.8)
axes[1].set_title('Yearly Average Temperature')
axes[1].set_ylabel('Temperature (°C)')
axes[1].xaxis.set_major_locator(mticker.MultipleLocator(5))

plt.savefig('../outputs/temperature_timeseries.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.2  Seasonal & distributional analysis ──────────────────────────────────
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Monthly box plot
monthly_data = [df_raw[df_raw['month'] == m]['mean_temp'].dropna().values
                for m in range(1, 13)]
bp = axes[0].boxplot(monthly_data, patch_artist=True, medianprops={'color': 'white', 'linewidth': 2})
colors_bp = sns.color_palette(PALETTE, 12)
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)
axes[0].set_xticklabels(month_labels, fontsize=8)
axes[0].set_title('Temperature Distribution by Month')
axes[0].set_ylabel('Mean Temp (°C)')

# Monthly average bar
monthly_avg = df_raw.groupby('month')['mean_temp'].mean()
bars = axes[1].bar(monthly_avg.index, monthly_avg.values,
                   color=sns.color_palette(PALETTE, 12), edgecolor='white', linewidth=0.5)
axes[1].bar_label(bars, fmt='%.1f°', padding=2, fontsize=7.5)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_labels, fontsize=8)
axes[1].set_title('Average Temperature by Month')
axes[1].set_ylabel('Mean Temp (°C)')

# Distribution (KDE + histogram)
axes[2].hist(df_raw['mean_temp'].dropna(), bins=45,
             color=ACCENT, edgecolor='white', alpha=0.7, density=True)
df_raw['mean_temp'].dropna().plot(kind='kde', ax=axes[2],
                                   color=HIGHLIGHT, linewidth=2)
axes[2].set_title('Distribution of Mean Temperature')
axes[2].set_xlabel('Temperature (°C)')
axes[2].set_ylabel('Density')

plt.tight_layout()
plt.savefig('../outputs/seasonal_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.3  Correlation analysis ────────────────────────────────────────────────
numeric_cols = df_raw.select_dtypes(include=np.number).drop(columns=['year', 'month'])
corr_matrix  = numeric_cols.corr()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Full heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, ax=axes[0], mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, linewidths=0.4,
            annot_kws={'size': 7.5}, square=True)
axes[0].set_title('Feature Correlation Matrix')

# Bar chart — correlation with target only
target_corr = (corr_matrix['mean_temp']
               .drop('mean_temp')
               .sort_values(key=abs, ascending=True))
colors_bar = [ACCENT if v >= 0 else HIGHLIGHT for v in target_corr.values]
axes[1].barh(target_corr.index, target_corr.values, color=colors_bar, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Correlation with Mean Temperature')
axes[1].set_xlabel('Pearson r')

plt.tight_layout()
plt.savefig('../outputs/correlation_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.4  Scatter plots — top correlated features ────────────────────────────
top_features = ['global_radiation', 'sunshine', 'max_temp', 'min_temp']

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, feat in zip(axes, top_features):
    ax.scatter(df_raw[feat], df_raw['mean_temp'],
               alpha=0.15, s=4, color=ACCENT)
    m, b = np.polyfit(df_raw[feat].dropna(),
                      df_raw.loc[df_raw[feat].notna(), 'mean_temp'], 1)
    x_line = np.linspace(df_raw[feat].min(), df_raw[feat].max(), 100)
    ax.plot(x_line, m * x_line + b, color=HIGHLIGHT, linewidth=1.8)
    ax.set_xlabel(feat.replace('_', ' ').title())
    ax.set_ylabel('Mean Temp (°C)' if feat == top_features[0] else '')
    ax.set_title(f'{feat.replace("_", " ").title()} vs Temp')

plt.suptitle('Key Feature Relationships with Mean Temperature', y=1.02, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/feature_scatter.png', bbox_inches='tight')
plt.show()

<a id='4'></a>
## 4 · Feature Engineering

Three categories of engineered features are added on top of the raw weather measurements:

| Category | Features | Rationale |
|---|---|---|
| **Cyclical time** | `month_sin/cos`, `doy_sin/cos` | Encodes periodicity so January and December are close |
| **Lag / rolling** | `lag_1`, `roll_3`, `roll_7` | Captures temperature momentum and recent trends |
| **Calendar** | `year`, `week_of_year` | Captures long-term climate drift |

In [ ]:
df = engineer_features(df_raw.copy())
df = clean_data(df)

print(f"Shape after engineering : {df.shape}")
df[['date', 'mean_temp', 'month_sin', 'month_cos',
    'doy_sin', 'doy_cos', 'mean_temp_lag1',
    'mean_temp_roll3', 'mean_temp_roll7']].head(8)

In [ ]:
# Visualise cyclical encoding
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sc1 = axes[0].scatter(df['month_sin'], df['month_cos'],
                      c=df['month'], cmap=PALETTE, s=8, alpha=0.6)
axes[0].set_title('Cyclical Month Encoding')
axes[0].set_xlabel('sin(month)')
axes[0].set_ylabel('cos(month)')
plt.colorbar(sc1, ax=axes[0], label='Month')

sc2 = axes[1].scatter(df['doy_sin'], df['doy_cos'],
                      c=df['day_of_year'], cmap=PALETTE, s=4, alpha=0.4)
axes[1].set_title('Cyclical Day-of-Year Encoding')
axes[1].set_xlabel('sin(day_of_year)')
axes[1].set_ylabel('cos(day_of_year)')
plt.colorbar(sc2, ax=axes[1], label='Day of Year')

plt.suptitle('Cyclical Feature Encoding — preserves circular distance between dates',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/cyclical_encoding.png', bbox_inches='tight')
plt.show()

In [ ]:
X, y = get_X_y(df)
print(f"Features : {X.shape[1]}")
print(f"Samples  : {X.shape[0]:,}")
print(f"Feature list: {X.columns.tolist()}")

<a id='5'></a>
## 5 · Train / Test Split

An **80 / 20 chronological split** is used — never shuffle time-series data, as future data would leak into training.

In [ ]:
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

dates = df.loc[X.index, 'date'] if 'date' in df.columns else df['date'].iloc[X.index]
train_dates = df['date'].iloc[:split]
test_dates  = df['date'].iloc[split:split + len(X_test)]

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(train_dates.values, y_train.values,
        color=ACCENT, linewidth=0.6, label=f'Train ({len(X_train):,} days)')
ax.plot(test_dates.values, y_test.values,
        color=HIGHLIGHT, linewidth=0.8, label=f'Test ({len(X_test):,} days)')
ax.axvline(train_dates.values[-1], color='black', linestyle='--',
           linewidth=1.2, label='Split point')
ax.set_title('Chronological Train / Test Split')
ax.set_ylabel('Mean Temp (°C)')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('../outputs/train_test_split.png', bbox_inches='tight')
plt.show()

print(f"Train: {train_dates.min().date()} → {train_dates.max().date()}  ({len(X_train):,} samples)")
print(f"Test : {test_dates.min().date()}  → {test_dates.max().date()}   ({len(X_test):,} samples)")

<a id='6'></a>
## 6 · Model Training & Experiment Tracking

Each model is wrapped in a **scikit-learn Pipeline** (StandardScaler → model) and evaluated with 5-fold `TimeSeriesSplit` cross-validation. All runs are logged to **MLflow**.

In [ ]:
mlflow.set_experiment('london_temperature_prediction')
tscv      = TimeSeriesSplit(n_splits=5)
pipelines = build_pipelines()
results   = []
all_preds = {}

for name, pipeline in pipelines.items():
    with mlflow.start_run(run_name=name):
        cv_scores = -cross_val_score(
            pipeline, X_train, y_train,
            cv=tscv, scoring='neg_root_mean_squared_error', n_jobs=-1
        )
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        all_preds[name] = y_pred

        row = {
            'Model':   name,
            'CV RMSE': round(cv_scores.mean(), 3),
            'CV Std':  round(cv_scores.std(),  3),
            'MAE':     round(mean_absolute_error(y_test, y_pred), 3),
            'RMSE':    round(np.sqrt(mean_squared_error(y_test, y_pred)), 3),
            'R²':      round(r2_score(y_test, y_pred), 4),
        }
        results.append(row)

        mlflow.log_params({'model': name})
        mlflow.log_metrics({
            'cv_rmse': row['CV RMSE'], 'cv_std': row['CV Std'],
            'test_mae': row['MAE'], 'test_rmse': row['RMSE'], 'test_r2': row['R²']
        })
        mlflow.sklearn.log_model(pipeline, artifact_path='model')

results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
results_df.style.background_gradient(subset=['RMSE', 'MAE'], cmap='RdYlGn_r') \
                .background_gradient(subset=['R²'], cmap='RdYlGn') \
                .format(precision=3)

<a id='7'></a>
## 7 · Model Comparison

In [ ]:
ordered   = results_df['Model'].tolist()
pal       = sns.color_palette('Set2', len(ordered))
color_map = dict(zip(ordered, pal))
bar_colors = [color_map[m] for m in results_df['Model']]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric, title in zip(axes,
                              ['RMSE', 'MAE', 'R²'],
                              ['RMSE  (↓ lower is better)',
                               'MAE   (↓ lower is better)',
                               'R²    (↑ higher is better)']):
    bars = ax.barh(results_df['Model'][::-1],
                   results_df[metric][::-1],
                   color=bar_colors[::-1], edgecolor='white')
    ax.bar_label(bars, fmt='%.3f', padding=4, fontsize=9)
    ax.set_title(title)
    ax.set_xlim(0, results_df[metric].max() * 1.22)
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle('Model Comparison — Test Set Performance', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# CV RMSE with error bars
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(results_df['Model'][::-1], results_df['CV RMSE'][::-1],
        xerr=results_df['CV Std'][::-1], color=bar_colors[::-1],
        edgecolor='white', capsize=4)
ax.set_title('Cross-Validation RMSE (5-fold TimeSeriesSplit) with Std Dev')
ax.set_xlabel('RMSE (°C)')
plt.tight_layout()
plt.savefig('../outputs/cv_comparison.png', bbox_inches='tight')
plt.show()

<a id='8'></a>
## 8 · Best Model Evaluation

In [ ]:
best_name     = results_df.iloc[0]['Model']
best_pipeline = pipelines[best_name]
y_pred_best   = all_preds[best_name]
residuals     = y_test.values - y_pred_best

print(f"Best model : {best_name}")
print(f"RMSE       : {results_df.iloc[0]['RMSE']} °C")
print(f"MAE        : {results_df.iloc[0]['MAE']} °C")
print(f"R²         : {results_df.iloc[0]['R²']}")

In [ ]:
# ── 8.1  Predicted vs Actual  ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
axes[0].scatter(y_test, y_pred_best, alpha=0.3, s=8, color=ACCENT)
lims = [min(y_test.min(), y_pred_best.min()) - 1,
        max(y_test.max(), y_pred_best.max()) + 1]
axes[0].plot(lims, lims, color=HIGHLIGHT, linewidth=1.8, linestyle='--', label='Perfect fit')
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel('Actual (°C)'); axes[0].set_ylabel('Predicted (°C)')
axes[0].set_title(f'{best_name}: Predicted vs Actual')
axes[0].legend(frameon=False)

# Time-series overlay (last 2 years)
n   = min(730, len(y_test))
idx = range(n)
axes[1].plot(idx, y_test.values[-n:],   color='#555', linewidth=1,   label='Actual', alpha=0.9)
axes[1].plot(idx, y_pred_best[-n:],     color=ACCENT, linewidth=1.2, label='Predicted', linestyle='--')
axes[1].fill_between(idx,
                     y_test.values[-n:],
                     y_pred_best[-n:],
                     alpha=0.15, color=HIGHLIGHT)
axes[1].set_title(f'{best_name}: Predictions vs Actuals (last 730 test days)')
axes[1].set_xlabel('Day'); axes[1].set_ylabel('Temperature (°C)')
axes[1].legend(frameon=False)

plt.tight_layout()
plt.savefig('../outputs/best_model_predictions.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.2  Residual analysis ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Residuals vs predicted
axes[0].scatter(y_pred_best, residuals, alpha=0.25, s=6, color=ACCENT)
axes[0].axhline(0, color=HIGHLIGHT, linewidth=1.5, linestyle='--')
axes[0].set_xlabel('Predicted (°C)'); axes[0].set_ylabel('Residual (°C)')
axes[0].set_title('Residuals vs Predicted')

# Residual distribution
axes[1].hist(residuals, bins=50, color=ACCENT, edgecolor='white', alpha=0.8, density=True)
pd.Series(residuals).plot(kind='kde', ax=axes[1], color=HIGHLIGHT, linewidth=2)
axes[1].axvline(0, color='black', linewidth=1, linestyle='--')
axes[1].set_xlabel('Residual (°C)'); axes[1].set_ylabel('Density')
axes[1].set_title('Residual Distribution')

# Residuals by month
test_months = df['month'].iloc[split:split + len(y_test)].values
res_by_month = pd.DataFrame({'month': test_months, 'residual': residuals})
monthly_res  = [res_by_month[res_by_month['month'] == m]['residual'].values
                for m in range(1, 13)]
bp2 = axes[2].boxplot(monthly_res, patch_artist=True,
                      medianprops={'color': 'white', 'linewidth': 1.8})
for patch, color in zip(bp2['boxes'], sns.color_palette(PALETTE, 12)):
    patch.set_facecolor(color)
axes[2].axhline(0, color=HIGHLIGHT, linewidth=1.2, linestyle='--')
axes[2].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'], fontsize=8)
axes[2].set_xlabel('Month'); axes[2].set_ylabel('Residual (°C)')
axes[2].set_title('Residuals by Month')

plt.suptitle(f'Residual Analysis — {best_name}', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/residual_analysis.png', bbox_inches='tight')
plt.show()

<a id='9'></a>
## 9 · Feature Importance

In [ ]:
model_step = best_pipeline.named_steps['model']

if hasattr(model_step, 'feature_importances_'):
    importances = pd.Series(model_step.feature_importances_, index=X.columns)
    importances = importances.sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(9, 6))
    colors_fi = [HIGHLIGHT if v > importances.median() else ACCENT
                 for v in importances.values]
    bars = ax.barh(importances.index, importances.values,
                   color=colors_fi, edgecolor='white')
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
    ax.set_xlabel('Importance')
    ax.set_title(f'Feature Importances — {best_name}')
    ax.set_xlim(0, importances.max() * 1.18)
    plt.tight_layout()
    plt.savefig('../outputs/feature_importance.png', bbox_inches='tight')
    plt.show()
else:
    print(f'{best_name} does not expose feature importances directly.')

<a id='10'></a>
## 10 · Summary

---

In [ ]:
best_row = results_df.iloc[0]
print("=" * 50)
print(" LONDON TEMPERATURE PREDICTION — RESULTS")
print("=" * 50)
print(f"  Best model : {best_row['Model']}")
print(f"  Test RMSE  : {best_row['RMSE']} °C")
print(f"  Test MAE   : {best_row['MAE']} °C")
print(f"  R²         : {best_row['R²']}")
print("=" * 50)
print()
print("All model results (sorted by RMSE):")
print(results_df.to_string(index=False))

---

### Key Findings

- **Lag features** (`mean_temp_lag1`, `roll_7`) were the strongest predictors, confirming that temperature has strong day-to-day autocorrelation.
- **Cyclical encoding** of month and day-of-year outperformed raw integer month as a feature, preserving seasonal continuity across year boundaries.
- **Gradient Boosting** achieved the lowest RMSE and highest R², demonstrating the value of non-linear ensemble methods for this task.
- Residual analysis shows the model performs consistently across all months with no strong seasonal bias.